In [0]:
# ============================================================
# SILVER LAYER - RETAIL DATA ENGINEERING PROJECT
# ============================================================

from pyspark.sql.functions import col, trim, upper


# ============================================================
# 1. PATHS
# ============================================================

silver_base_path = (
    "abfss://silver@jadabalastorageaccount.dfs.core.windows.net"
)

silver_customers_path = f"{silver_base_path}/customers"
silver_products_path = f"{silver_base_path}/products"
silver_stores_path = f"{silver_base_path}/stores"
silver_employees_path = f"{silver_base_path}/employees"
silver_orders_path = f"{silver_base_path}/orders"
silver_order_items_path = f"{silver_base_path}/order_items"


# ============================================================
# 2. READ BRONZE TABLES
# ============================================================

bronze_customers_df = spark.table(
    "retail_catalog.bronze.customers"
)

bronze_products_df = spark.table(
    "retail_catalog.bronze.products"
)

bronze_stores_df = spark.table(
    "retail_catalog.bronze.stores"
)

bronze_employees_df = spark.table(
    "retail_catalog.bronze.employees"
)

bronze_orders_df = spark.table(
    "retail_catalog.bronze.orders"
)

bronze_order_items_df = spark.table(
    "retail_catalog.bronze.order_items"
)


# ============================================================
# 3. CUSTOMERS
# ============================================================

# ----------------------------
# Clean / Standardize
# ----------------------------

customers_clean_df = (
    bronze_customers_df
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
    .withColumn("city", upper(trim(col("city"))))
    .withColumn("state", upper(trim(col("state"))))
    .withColumn("country", upper(trim(col("country"))))
)

# ----------------------------
# Invalid Customers
# ----------------------------

invalid_customers_df = (
    customers_clean_df
    .filter(col("customer_id").isNull())
)

# ----------------------------
# Valid / Silver Customers
# ----------------------------

silver_customers_df = (
    customers_clean_df
    .filter(col("customer_id").isNotNull())
    .dropDuplicates()
)


# ============================================================
# 4. PRODUCTS
# ============================================================

# ----------------------------
# Clean / Standardize
# ----------------------------

products_clean_df = (
    bronze_products_df
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("brand", trim(col("brand")))
    .withColumn("supplier", trim(col("supplier")))
)

# ----------------------------
# Invalid Products
# ----------------------------

invalid_products_df = (
    products_clean_df
    .filter(
        col("product_id").isNull() |
        (col("product_id") == "") |
        (col("unit_price") < 0)
    )
)

# ----------------------------
# Valid / Silver Products
# ----------------------------

silver_products_df = (
    products_clean_df
    .filter(col("product_id").isNotNull())
    .filter(col("product_id") != "")
    .filter(col("unit_price") >= 0)
    .dropDuplicates()
)


# ============================================================
# 5. STORES
# ============================================================

# ----------------------------
# Clean / Standardize
# ----------------------------

stores_clean_df = (
    bronze_stores_df
    .withColumn("store_id", trim(col("store_id")))
    .withColumn("store_name", upper(trim(col("store_name"))))
    .withColumn("city", upper(trim(col("city"))))
    .withColumn("state", upper(trim(col("state"))))
    .withColumn("country", upper(trim(col("country"))))
    .withColumn("store_address", upper(trim(col("store_address"))))
    .withColumn("phone", col("phone").cast("string"))
)

# ----------------------------
# Invalid Stores
# ----------------------------

invalid_stores_df = (
    stores_clean_df
    .filter(
        col("store_id").isNull() |
        (col("store_id") == "")
    )
)

# ----------------------------
# Valid / Silver Stores
# ----------------------------

silver_stores_df = (
    stores_clean_df
    .filter(col("store_id").isNotNull())
    .filter(col("store_id") != "")
    .dropDuplicates()
)


# ============================================================
# 6. EMPLOYEES
# ============================================================

# ----------------------------
# Clean / Standardize
# ----------------------------

employees_clean_df = (
    bronze_employees_df
    .withColumn("employee_id", trim(col("employee_id")))
    .withColumn("employee_name", trim(col("employee_name")))
    .withColumn("designation", trim(col("designation")))
    .withColumn("store_id", trim(col("store_id")))
)

# ----------------------------
# Basic Employee Validation
# ----------------------------

basic_valid_employees_df = (
    employees_clean_df
    .filter(col("employee_id").isNotNull())
    .filter(col("employee_id") != "")
    .filter(col("store_id").isNotNull())
    .filter(col("store_id") != "")
    .filter(col("salary") > 0)
    .dropDuplicates()
)

# ----------------------------
# Invalid Store References
# ----------------------------

invalid_store_employees_df = (
    basic_valid_employees_df
    .join(
        silver_stores_df,
        on="store_id",
        how="left_anti"
    )
)

# ----------------------------
# Valid / Silver Employees
# ----------------------------

silver_employees_df = (
    basic_valid_employees_df
    .join(
        silver_stores_df,
        on="store_id",
        how="left_semi"
    )
)


# ============================================================
# 7. ORDERS
# ============================================================

# IMPORTANT:
# order_id must be STRING in Bronze because the source contains
# alphanumeric IDs such as O00001.
#
# Bronze must have been recreated using the corrected schema
# before running this section.


# ----------------------------
# Clean / Standardize
# ----------------------------

orders_clean_df = (
    bronze_orders_df
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("store_id", trim(col("store_id")))
    .withColumn(
        "payment_mode",
        upper(trim(col("payment_mode")))
    )
    .withColumn(
        "order_status",
        upper(trim(col("order_status")))
    )
)

# ----------------------------
# Basic Order Validation
# ----------------------------

basic_valid_orders_df = (
    orders_clean_df
    .filter(col("order_id").isNotNull())
    .filter(col("order_id") != "")
    .filter(col("customer_id").isNotNull())
    .filter(col("store_id").isNotNull())
    .filter(col("store_id") != "")
    .filter(col("order_date").isNotNull())
    .filter(col("payment_mode").isNotNull())
    .filter(col("payment_mode") != "")
    .filter(col("order_status").isNotNull())
    .filter(col("order_status") != "")
    .dropDuplicates()
)

# ----------------------------
# Invalid Customer References
# ----------------------------

invalid_customer_orders_df = (
    basic_valid_orders_df
    .join(
        silver_customers_df,
        on="customer_id",
        how="left_anti"
    )
)

# ----------------------------
# Invalid Store References
# ----------------------------

invalid_store_orders_df = (
    basic_valid_orders_df
    .join(
        silver_stores_df,
        on="store_id",
        how="left_anti"
    )
)

# ----------------------------
# Valid Customer References
# ----------------------------

orders_with_valid_customers_df = (
    basic_valid_orders_df
    .join(
        silver_customers_df,
        on="customer_id",
        how="left_semi"
    )
)

# ----------------------------
# Valid Customer + Store
# ----------------------------

silver_orders_df = (
    orders_with_valid_customers_df
    .join(
        silver_stores_df,
        on="store_id",
        how="left_semi"
    )
)


# ============================================================
# 8. ORDER ITEMS
# ============================================================

# ----------------------------
# Clean / Standardize
# ----------------------------

order_items_clean_df = (
    bronze_order_items_df
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("product_id", trim(col("product_id")))
)

# ----------------------------
# Basic Validation
# ----------------------------

basic_valid_order_items_df = (
    order_items_clean_df
    .filter(col("order_item_id").isNotNull())
    .filter(col("order_id").isNotNull())
    .filter(col("order_id") != "")
    .filter(col("product_id").isNotNull())
    .filter(col("product_id") != "")
    .filter(col("quantity") > 0)
    .filter(col("selling_price") >= 0)
    .dropDuplicates()
)

# ----------------------------
# Invalid Order References
# ----------------------------

invalid_order_reference_items_df = (
    basic_valid_order_items_df
    .join(
        silver_orders_df,
        on="order_id",
        how="left_anti"
    )
)

# ----------------------------
# Invalid Product References
# ----------------------------

invalid_product_reference_items_df = (
    basic_valid_order_items_df
    .join(
        silver_products_df,
        on="product_id",
        how="left_anti"
    )
)

# ----------------------------
# Valid Order References
# ----------------------------

items_with_valid_orders_df = (
    basic_valid_order_items_df
    .join(
        silver_orders_df,
        on="order_id",
        how="left_semi"
    )
)

# ----------------------------
# Valid Order + Product
# ----------------------------

silver_order_items_df = (
    items_with_valid_orders_df
    .join(
        silver_products_df,
        on="product_id",
        how="left_semi"
    )
)


# ============================================================
# 9. WRITE SILVER DELTA DATA TO ADLS
# ============================================================

silver_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_customers_path)

silver_products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_products_path)

silver_stores_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_stores_path)

silver_employees_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_employees_path)

silver_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_orders_path)

silver_order_items_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_order_items_path)


# ============================================================
# 10. CREATE / REGISTER UNITY CATALOG EXTERNAL TABLES
# ============================================================

spark.sql("""
CREATE TABLE IF NOT EXISTS retail_catalog.silver.customers
USING DELTA
LOCATION 'abfss://silver@jadabalastorageaccount.dfs.core.windows.net/customers'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS retail_catalog.silver.products
USING DELTA
LOCATION 'abfss://silver@jadabalastorageaccount.dfs.core.windows.net/products'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS retail_catalog.silver.stores
USING DELTA
LOCATION 'abfss://silver@jadabalastorageaccount.dfs.core.windows.net/stores'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS retail_catalog.silver.employees
USING DELTA
LOCATION 'abfss://silver@jadabalastorageaccount.dfs.core.windows.net/employees'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS retail_catalog.silver.orders
USING DELTA
LOCATION 'abfss://silver@jadabalastorageaccount.dfs.core.windows.net/orders'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS retail_catalog.silver.order_items
USING DELTA
LOCATION 'abfss://silver@jadabalastorageaccount.dfs.core.windows.net/order_items'
""")


# ============================================================
# 11. VERIFY SILVER TABLES
# ============================================================

spark.table("retail_catalog.silver.customers").show()
spark.table("retail_catalog.silver.products").show()
spark.table("retail_catalog.silver.stores").show()
spark.table("retail_catalog.silver.employees").show()
spark.table("retail_catalog.silver.orders").show()
spark.table("retail_catalog.silver.order_items").show()

+-----------+----------+---------+------+--------------------+----------+---------+-----------+-------+-----------+
|customer_id|first_name|last_name|gender|               email|     phone|     city|      state|country|signup_date|
+-----------+----------+---------+------+--------------------+----------+---------+-----------+-------+-----------+
|       1016|  Cust1016|     User|     M|cust1016@example.com|9349957310|  CHENNAI| TAMIL NADU|  INDIA| 2025-01-26|
|       1019|  Cust1019|     User|     F|cust1019@example.com|9853573823|BANGALORE|  KARNATAKA|  INDIA| 2025-06-24|
|       1023|  Cust1023|     User|     M|cust1023@example.com|9982403818|HYDERABAD|  TELANGANA|  INDIA| 2025-01-09|
|       1028|  Cust1028|     User|     F|cust1028@example.com|9902099969|     PUNE|MAHARASHTRA|  INDIA| 2025-05-30|
|       1035|  Cust1035|     User|     F|cust1035@example.com|9925276600|     PUNE|MAHARASHTRA|  INDIA| 2025-06-14|
|       1040|  Cust1040|     User|     M|cust1040@example.com|9743111853